# Turbiner's Riccati-Bloch Nonlinearization

## Log-Wavefunction Methods in Quantum Spectral Theory

**Author:** Keenan M Stone  
**This notebook:** April 2026  
**Status:** Review of Turbiner's method, computational benchmarks, and identification of limitations

---

### Purpose

This notebook reviews **Turbiner's 1984 nonlinearization** — the transformation of the Schrödinger equation into a Riccati-Bloch equation via the log-wavefunction substitution $y = -\psi'/\psi$. We:

1. Present the theoretical framework and its place in quantum spectral theory
2. Benchmark Riccati-based eigenvalue detection against standard methods
3. Test whether the Riccati approach has computational advantages for specific problem classes
4. Explore the transfer operator connection (Frobenius-Perron, Ruelle-Pollicott resonances)
5. Visualize "quantization as regularity" via Riccati regularity maps
6. Identify knowledge gaps and dead ends

**Companion notebooks:** See [alpha_transform.ipynb](alpha_transform.ipynb) for the α-transform theory, and [crossover_alpha_turbiner.ipynb](crossover_alpha_turbiner.ipynb) for where both methods intersect.

## 1. Turbiner's Nonlinearization: Theory

### 1.1 The Substitution

The time-independent Schrödinger equation (in units where $\hbar = 2m = 1$):

$$-\psi''(x) + 2V(x)\psi(x) = 2E\psi(x)$$

Define the **log-derivative** $y(x) = -\psi'(x)/\psi(x)$. Then:

$$y'(x) = y(x)^2 - 2V(x) + 2E$$

This is the **Riccati-Bloch equation** — a nonlinear first-order ODE. The key insight [Turbiner 1984]:

> **Quantization condition:** $y(x)$ is globally bounded on $\mathbb{R}$ (or $[0,\infty)$ for radial problems) if and only if $E$ is an eigenvalue of the original Schrödinger equation.

At non-eigenvalues, $y(x)$ develops a **Riccati singularity** — it diverges in finite "time" (spatial extent) because the $y^2$ nonlinearity causes blow-up when the trajectory escapes the basin of bounded solutions.

### 1.2 Physical Interpretation

The log-derivative $y(x) = -\psi'/\psi$ has a direct physical meaning:

- In the **classically allowed** region ($V < E$): $y(x)$ oscillates, encoding the local momentum $k(x) = \sqrt{2(E-V)}$ via $y \sim k\tan(\int k\,dx)$
- In the **classically forbidden** region ($V > E$): $y(x) \sim \kappa = \sqrt{2(V-E)}$, encoding the exponential decay rate
- At **nodes** of $\psi$: $y(x) \to \pm\infty$ (poles of the log-derivative)
- The **ground state** (nodeless $\psi_0$) has globally bounded $y(x)$ with no poles

### 1.3 Connection to WKB and Semiclassical Methods

The WKB approximation treats $y(x) \approx \sqrt{2(V-E)}$ as the zeroth-order solution. The exact Riccati equation $y' = y^2 - 2V + 2E$ includes the **quantum correction** $y'$ (the derivative term that WKB neglects). Turbiner's contribution was recognizing that:

1. Certain potentials (quasi-exactly-solvable, QES) allow $y(x)$ to be polynomial
2. This yields exact eigenvalues via algebraic equations
3. The regularity condition is the **nonlinear analog** of square-integrability

### 1.4 What Turbiner's Method Has Been Used For

- **Quasi-exactly-solvable (QES) models** [Turbiner 1988]: Classification of potentials where a finite number of eigenvalues can be found exactly via the Lie-algebraic structure of the Riccati equation
- **Asymptotic expansion of eigenvalues** [Turbiner & Ushveridze 1987]: The Riccati equation as an organizing principle for perturbation theory
- **Semiclassical quantization** [Bender & Orszag]: The Riccati equation as the nonlinear analog of the Bohr-Sommerfeld condition
- **WKB corrections**: Systematic improvement beyond leading-order semiclassical via iterating the Riccati equation
- **PT-symmetric quantum mechanics** [Bender et al.]: Extension to complex potentials where the Riccati equation has complex $y(x)$

### 1.5 Knowledge Gaps

Despite extensive analytic use, several questions remain underexplored:

1. **Computational implementation:** Has anyone built practical eigenvalue solvers around the Riccati-Bloch equation, or is it exclusively an analytical/asymptotic tool?
2. **Transfer operator connection:** Can the Riccati equation define a spectral problem on phase space that differs from the Frobenius-Perron operator?
3. **Singular potentials:** Does the Riccati formulation handle Coulomb-type singularities better or worse than standard shooting methods?
4. **Connection to relaxation methods:** Can the α-transform (see companion notebook) stabilize Riccati trajectories for excited states?

### 1.6 References

1. **Turbiner, A.V.** "Quasi-exactly-solvable problems and sl(2) algebra." *Commun. Math. Phys.* 118 (1988): 467.
2. **Turbiner, A.V.** "One-dimensional quasi-exactly solvable Schrödinger equations." *Phys. Rep.* 642 (2016): 1–71.
3. **Turbiner, A.V. and Ushveridze, A.G.** "Anharmonic oscillator: constructing the strong coupling expansions." *J. Math. Phys.* 29 (1988): 2053.
4. **Bender, C.M. and Orszag, S.A.** *Advanced Mathematical Methods for Scientists and Engineers.* Springer, 1999.
5. **Reid, W.T.** *Riccati Differential Equations.* Academic Press, 1972.
6. **Ruelle, D.** "Resonances of chaotic dynamical systems." *Phys. Rev. Lett.* 56 (1986): 405.
7. **Froyland, G.** "Approximating physical invariant measures of mixing dynamical systems." *Nonlinearity* 11 (1998): 1043.
8. **Blank, M., Keller, G., and Liverani, C.** "Ruelle-Perron-Frobenius spectrum for Anosov maps." *Nonlinearity* 15 (2002): 1905.
9. **Mezić, I.** "Spectral properties of dynamical systems, model reduction and decompositions." *Nonlinear Dynamics* 41 (2005): 309.

## 2. Setup and Imports

In [ ]:
import sys, time, importlib
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.integrate import solve_ivp

from petrification.maps import logistic
from petrification.potentials import harmonic, anharmonic, double_well, morse, coulomb
from petrification.quantum import (
    discretize_hamiltonian, solve_eigenstates,
    riccati_solve, riccati_alpha,
    spectral_scan, detect_eigenvalues,
    numerov_detect, numerov_shoot, numerov_scan,
    frobenius_perron_matrix, exact_ulam_matrix,
    chebyshev_transfer_matrix, fp_spectrum_at_params,
    riccati_regularity_map,
)

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 8), 'font.size': 12})
print('All imports successful.')

## 3. Riccati-Bloch as a Dynamical System

### 3.1 Spectral Scanning

Eigenvalues appear as sharp dips in $|\psi(x_{\max}, E)|$ — these are the energies where the shooting function has a zero.

In [ ]:
# Reference eigenvalues via matrix diagonalization
N = 500
x_grid = np.linspace(-8, 8, N)
eigenvalues, eigenvectors = solve_eigenstates(harmonic, x_grid, n_states=8)
evals_full = eigh(discretize_hamiltonian(harmonic, x_grid), eigvals_only=True)

# Spectral scan
E_range = np.linspace(0.01, 8.5, 2000)
x_span = (0.0, 7.0)

psi_even = spectral_scan(harmonic, E_range, x_span, parity='even')
psi_odd = spectral_scan(harmonic, E_range, x_span, parity='odd')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(E_range, np.log10(np.abs(psi_even) + 1e-20), 'steelblue', lw=1.0, label='Even parity')
ax.plot(E_range, np.log10(np.abs(psi_odd) + 1e-20), '#ff7f0e', lw=1.0, label='Odd parity')

for i in range(8):
    ev = evals_full[i]
    if ev <= 8.5:
        ax.axvline(ev, color='red', alpha=0.4, lw=0.8, ls='--')

ax.set_xlabel('Trial energy $E$')
ax.set_ylabel(r'$\log_{10}|\psi(x_{\max}, E)|$')
ax.set_title('Spectral shooting: sharp dips indicate eigenvalues')
ax.legend()
ax.set_ylim(-18, 20)
plt.tight_layout()
plt.show()

### 3.2 Riccati Trajectories: Bounded iff $E = E_n$

The Riccati-Bloch equation $y' = y^2 - 2V + 2E$ gives bounded $y(x)$ only at eigenvalues.

In [ ]:
E_exact = evals_full[0]

offsets = [-0.3, -0.1, -0.01, 0.0, 0.01, 0.1, 0.3]
colors_r = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#bcbd22', '#ff7f0e', '#d62728']
widths = [0.8, 1.0, 1.2, 2.5, 1.2, 1.0, 0.8]

fig, ax = plt.subplots(figsize=(10, 6))
for offset, color, w in zip(offsets, colors_r, widths):
    E_trial = E_exact + offset
    sol = riccati_solve(harmonic, E_trial, (0.01, 6.0), y0=0.01)
    label = rf'$E = E_0 {"+ " if offset >= 0 else ""}{offset}$'
    ax.plot(sol.t, sol.y[0], color=color, lw=w, label=label)

ax.set_xlabel('$x$')
ax.set_ylabel(r"$y(x) = -\psi'/\psi$")
ax.set_title(f'Riccati-Bloch trajectories near $E_0 = {E_exact:.4f}$\n'
             'Only the green trajectory (at eigenvalue) stays bounded')
ax.set_ylim(-20, 20)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 3.3 Alpha-Relaxed Riccati

The alpha-transform can be applied to the discretized Riccati equation:

$$y_{n+1} = \alpha\left[y_n + h\left(y_n^2 - 2V(x_n) + 2E\right)\right] + (1-\alpha)y_n$$

The hypothesis: different $\alpha$ values can stabilize Riccati trajectories for excited states.

In [ ]:
E1 = evals_full[1]
x_alpha_grid = np.linspace(0.01, 5.0, 1500)

alphas_demo = [0.3, 0.5, 0.7, 1.0, 1.3]
colors_a = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, ax = plt.subplots(figsize=(10, 6))
for alpha_val, color in zip(alphas_demo, colors_a):
    y_traj = riccati_alpha(harmonic, E1, x_alpha_grid, y0=0.01, alpha=alpha_val)
    ax.plot(x_alpha_grid, y_traj, color=color, lw=1.3, label=rf'$\alpha = {alpha_val}$')

ax.set_xlabel('$x$')
ax.set_ylabel('$y(x)$')
ax.set_title(rf'Alpha-relaxed Riccati at $E_1 = {E1:.4f}$ (first excited state)' + '\n'
             r'$\alpha$ controls the stability of the nonlinear trajectory')
ax.set_ylim(-20, 20)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Eigenvalue Detection

### 4.1 Automated Detection for Harmonic and Anharmonic Oscillators

In [ ]:
# Harmonic oscillator
E_fine = np.linspace(0.01, 8.0, 5000)
E_detected = detect_eigenvalues(harmonic, E_fine, x_span=(0.0, 7.0), order=20)

print("Eigenvalue detection via Riccati/shooting fixed-point method:")
print(f"{'Detected':>12}  {'Exact (matrix)':>14}  {'Exact (analytic)':>16}  {'Error':>10}")
for E_det in E_detected:
    near_idx = np.argmin(np.abs(evals_full - E_det))
    E_mat = evals_full[near_idx]
    E_ana = near_idx + 0.5
    print(f"{E_det:12.5f}  {E_mat:14.5f}  {E_ana:16.1f}  {abs(E_det - E_ana):10.2e}")

In [ ]:
# Anharmonic oscillator — no analytic solution
V_anh = lambda x: anharmonic(x, g=0.1)
evals_anh = eigh(discretize_hamiltonian(V_anh, x_grid), eigvals_only=True)

E_anh_range = np.linspace(0.01, 12.0, 3000)
E_anh_detected = detect_eigenvalues(V_anh, E_anh_range, x_span=(0.0, 7.0), order=10)

psi_anh_even = spectral_scan(V_anh, E_anh_range, (0.0, 7.0), parity='even')
psi_anh_odd = spectral_scan(V_anh, E_anh_range, (0.0, 7.0), parity='odd')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(E_anh_range, np.log10(np.abs(psi_anh_even) + 1e-20), 'steelblue', lw=0.8, label='Even')
ax.plot(E_anh_range, np.log10(np.abs(psi_anh_odd) + 1e-20), '#ff7f0e', lw=0.8, label='Odd')
for i in range(6):
    ev = evals_anh[i]
    if ev <= 12.0:
        ax.axvline(ev, color='red', alpha=0.5, lw=0.8, ls='--')
ax.set_xlabel('Trial energy $E$')
ax.set_ylabel(r'$\log_{10}|\psi(x_{\max})|$')
ax.set_title('Anharmonic oscillator $V = x^2 + 0.1x^4$: eigenvalues from shooting')
ax.legend()
ax.set_ylim(-18, 20)
plt.tight_layout()
plt.show()

print("\nAnharmonic oscillator eigenvalue detection:")
print(f"{'Detected':>12}  {'Matrix diag':>12}  {'Error':>10}")
for E_det in E_anh_detected:
    near_idx = np.argmin(np.abs(evals_anh - E_det))
    print(f"{E_det:12.4f}  {evals_anh[near_idx]:12.4f}  {abs(E_det - evals_anh[near_idx]):10.2e}")

## 5. Computational Benchmarks

### Experiment 1: Speed — Riccati-Shooting vs Numerov vs Matrix Diagonalization

**Question:** Does the Riccati-Bloch shooting method have any speed/accuracy advantage over standard methods?

In [ ]:
potentials = {
    'Harmonic': (harmonic, (-8, 8), (0, 7), 8),
    'Anharmonic': (lambda x: anharmonic(x, g=0.1), (-8, 8), (0, 7), 7),
    'Double well': (double_well, (-4, 4), (0, 4), 4),
    'Morse': (lambda x: morse(x, D=10, alpha=1.0, x0=0.0), (-2, 8), (0, 6), 5),
}

grid_sizes = [200, 500, 1000]
E_scan = np.linspace(0.01, 12.0, 2000)

print("=" * 80)
print("EXPERIMENT 1: Speed & Accuracy Benchmark")
print("=" * 80)

for pot_name, (V, x_lim, x_span, n_expect) in potentials.items():
    print(f"\n{'\u2500' * 60}")
    print(f"  {pot_name} potential")
    print(f"{'\u2500' * 60}")
    
    for N_bench in grid_sizes:
        x_g = np.linspace(x_lim[0], x_lim[1], N_bench)
        x_shoot = np.linspace(x_span[0], x_span[1], N_bench)
        x_ref = np.linspace(x_lim[0], x_lim[1], 2000)
        E_ref = eigh(discretize_hamiltonian(V, x_ref), eigvals_only=True)[:n_expect]
        
        # Method 1: Matrix diagonalization
        t0 = time.perf_counter()
        E_mat = eigh(discretize_hamiltonian(V, x_g), eigvals_only=True)[:n_expect]
        t_mat = time.perf_counter() - t0
        
        # Method 2: Numerov + bisection
        t0 = time.perf_counter()
        E_num = numerov_detect(V, E_scan, x_shoot, order=15)[:n_expect]
        t_num = time.perf_counter() - t0
        
        # Method 3: Riccati-Bloch shooting
        t0 = time.perf_counter()
        E_ric = detect_eigenvalues(V, E_scan, x_span=x_span, order=15)[:n_expect]
        t_ric = time.perf_counter() - t0
        
        def max_err(E_found, E_true):
            if len(E_found) == 0: return float('inf')
            return np.max([min(abs(e - E_true)) for e in E_found])
        
        print(f"\n  N = {N_bench}:")
        print(f"    {'Method':<22} {'Time (s)':>10} {'Max Error':>12} {'Found':>6}")
        print(f"    {'Matrix diag':<22} {t_mat:10.4f} {max_err(E_mat, E_ref):12.2e} {n_expect:6d}")
        print(f"    {'Numerov+bisect':<22} {t_num:10.4f} {max_err(E_num, E_ref):12.2e} {len(E_num):6d}")
        print(f"    {'Riccati-Bloch shoot':<22} {t_ric:10.4f} {max_err(E_ric, E_ref):12.2e} {len(E_ric):6d}")

### Experiment 1 Result: No Advantage for Riccati

| Method | Time (harmonic, N=500) | Max Error |
|--------|----------------------|-----------|
| Matrix diag | ~0.08s | ~3e-3 |
| Numerov+bisection | ~2.8s | ~2e-4 |
| Riccati-Bloch | ~92s | ~3e-3 |

The Riccati-Bloch shooting method is **~1200x slower** than matrix diag with no accuracy gain. The RK45 adaptive integrator is expensive per energy trial, and scanning 2000 trial energies compounds this.

### Experiment 2: Alpha-Relaxed Riccati for Tunnel Splitting

**Question:** Can alpha-relaxation help detect quasi-degenerate eigenvalue pairs (tunnel splittings) in double-well potentials?

In [ ]:
V_dw = lambda x: double_well(x, a=1.0)

# Ground truth
x_truth = np.linspace(-6, 6, 3000)
E_truth = eigh(discretize_hamiltonian(V_dw, x_truth), eigvals_only=True)[:8]

print("Double-well eigenvalues (ground truth, N=3000):")
for i, e in enumerate(E_truth):
    pair_label = f"  pair {i//2}" if i % 2 == 0 else f"  pair {i//2}  \u0394={e - E_truth[i-1]:.6e}"
    print(f"  E_{i} = {e:.8f}{pair_label}")

# Alpha-relaxed Riccati: does damping help resolve splittings?
x_ric_grid = np.linspace(0, 5, 500)
E_fine_dw = np.linspace(-0.1, 5.0, 2000)
alpha_test = [0.2, 0.4, 0.6, 0.8, 1.0]

fig, axes = plt.subplots(len(alpha_test), 1, figsize=(12, 3*len(alpha_test)), sharex=True)

for idx, alpha_r in enumerate(alpha_test):
    divergence_measure = np.zeros(len(E_fine_dw))
    for j, E_trial in enumerate(E_fine_dw):
        traj = riccati_alpha(V_dw, E_trial, x_ric_grid, y0=0.0, alpha=alpha_r, max_y=1e6)
        bounded = np.sum(np.abs(traj) < 100)
        divergence_measure[j] = bounded / len(x_ric_grid)
    
    axes[idx].plot(E_fine_dw, divergence_measure, 'b-', lw=0.8)
    for e in E_truth[:6]:
        axes[idx].axvline(e, color='red', alpha=0.4, lw=0.8, ls='--')
    axes[idx].set_ylabel(f'\u03b1={alpha_r}')
    axes[idx].set_ylim(0, 1.1)

axes[-1].set_xlabel('Trial energy $E$')
axes[0].set_title('Alpha-relaxed Riccati: fraction of trajectory that stays bounded\n'
                   'Lower \u03b1 = wider basin but less discrimination')
plt.tight_layout()
plt.show()

# Basin width measurement
print("\nBasin width around ground state for each alpha:")
E_gs = E_truth[0]
E_fine_zoom = np.linspace(E_gs - 0.5, E_gs + 0.5, 1000)

for alpha_r in alpha_test:
    bounded_count = 0
    for E_trial in E_fine_zoom:
        traj = riccati_alpha(V_dw, E_trial, x_ric_grid, y0=0.0, alpha=alpha_r, max_y=1e6)
        if np.all(np.abs(traj) < 100):
            bounded_count += 1
    basin_fraction = bounded_count / len(E_fine_zoom)
    basin_width = basin_fraction * (E_fine_zoom[-1] - E_fine_zoom[0])
    print(f"  \u03b1 = {alpha_r}: {bounded_count}/{len(E_fine_zoom)} bounded "
          f"\u2192 basin width \u2248 {basin_width:.4f}")

### Experiment 2 Result: Real Effect, But Wrong Tradeoff

Lower $\alpha$ genuinely widens the basin of stability for Riccati trajectories. But wider basins = less discrimination. At $\alpha = 0.2$, everything looks bounded — you can't tell eigenvalues from non-eigenvalues. The eigenvalue-detection signal comes from *sharp* divergence, and damping destroys this sharpness.

**Verdict:** $\alpha$-relaxation trades sensitivity for stability — the wrong tradeoff for eigenvalue detection.

## 6. Transfer Operators: From Quantum to Dynamical Systems

### Experiment 3: Frobenius-Perron Operator for the Logistic Map

The **Frobenius-Perron (transfer) operator** $\mathcal{L}$ propagates probability densities under a map: $(\mathcal{L}\rho)(x) = \sum_{y: f(y)=x} \rho(y)/|f'(y)|$.

Predictions to test:
1. Leading eigenvalue = 1 (conservation of probability)
2. Spectral gap encodes the mixing rate (decay of correlations)
3. Spectrum shows structure at bifurcation points
4. At $a=4$, the eigenvector matches the arcsine distribution $\rho(x) = 1/[\pi\sqrt{x(1-x)}]$

In [ ]:
# Frobenius-Perron operator at a=4
N_fp = 300
L_4, bins_4 = frobenius_perron_matrix(logistic, 4.0, N=N_fp)

evals_L, evecs_L = np.linalg.eig(L_4)
order = np.argsort(np.abs(evals_L))[::-1]
evals_L = evals_L[order]
evecs_L = evecs_L[:, order]

print(f"Logistic map a=4, N={N_fp} bins")
print(f"Top 10 eigenvalues of Frobenius-Perron operator:")
for i in range(10):
    ev = evals_L[i]
    print(f"  \u03bb_{i} = {ev.real:+.6f} {ev.imag:+.6f}j  |\u03bb| = {abs(ev):.6f}")

# Invariant measure comparison
rho_computed = np.abs(np.real(evecs_L[:, 0]))
rho_computed /= np.sum(rho_computed) * (bins_4[1] - bins_4[0])
rho_exact = 1.0 / (np.pi * np.sqrt(bins_4 * (1 - bins_4) + 1e-30))
rho_exact /= np.sum(rho_exact) * (bins_4[1] - bins_4[0])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(bins_4, rho_computed, 'b-', lw=1.5, label='FP eigenvector')
axes[0].plot(bins_4, rho_exact, 'r--', lw=1.5, label=r'$\frac{1}{\pi\sqrt{x(1-x)}}$')
axes[0].set_xlabel('$x$'); axes[0].set_ylabel(r'$\rho(x)$')
axes[0].set_title('Invariant measure at $a=4$')
axes[0].legend(); axes[0].set_ylim(0, 8)

axes[1].scatter(np.real(evals_L[:50]), np.imag(evals_L[:50]),
                s=20, c=np.arange(50), cmap='viridis')
theta = np.linspace(0, 2*np.pi, 100)
axes[1].plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3)
axes[1].set_xlabel('Re(\u03bb)'); axes[1].set_ylabel('Im(\u03bb)')
axes[1].set_title('FP eigenvalue spectrum'); axes[1].set_aspect('equal')

# FP spectrum across Feigenbaum cascade
a_values = np.linspace(2.5, 4.0, 80)
top_evals = np.zeros((len(a_values), 8), dtype=complex)
for i, a_val in enumerate(a_values):
    L_a, _ = frobenius_perron_matrix(logistic, a_val, N=150)
    ev_a = np.linalg.eigvals(L_a)
    ev_sorted = ev_a[np.argsort(np.abs(ev_a))[::-1]]
    top_evals[i, :] = ev_sorted[:8]

for k in range(8):
    axes[2].scatter(a_values, np.abs(top_evals[:, k]), s=3, alpha=0.6)
axes[2].axvline(3.0, color='gray', alpha=0.3, ls='--', label='Period-2')
axes[2].axvline(3.449, color='gray', alpha=0.3, ls=':')
axes[2].axvline(3.5699, color='gray', alpha=0.3, ls='-.')
axes[2].set_xlabel('Parameter $a$'); axes[2].set_ylabel('$|\\lambda_k|$')
axes[2].set_title('FP spectrum across Feigenbaum cascade')
axes[2].set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

spectral_gap = abs(evals_L[0]) - abs(evals_L[1])
print(f"\nSpectral gap: {spectral_gap:.4f}")
print(f"Known Lyapunov exponent at a=4: ln(2) = {np.log(2):.6f}")

### Experiment 3b: Effective Potential from Invariant Measure

If we treat $\sqrt{\rho_{\text{inv}}}$ as a ground-state wavefunction, inverting the Schr\"odinger equation gives $V_{\text{eff}}(x) = \psi_0''/(2\psi_0)$.

In [ ]:
x_fine = np.linspace(0.01, 0.99, 500)
dx_fine = x_fine[1] - x_fine[0]
rho_inv = 1.0 / (np.pi * np.sqrt(x_fine * (1 - x_fine)))
psi_0 = np.sqrt(rho_inv)

psi_0_pp = np.gradient(np.gradient(psi_0, dx_fine), dx_fine)
V_eff = psi_0_pp / (2.0 * psi_0)
V_eff -= np.nanmin(V_eff[20:-20])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(x_fine[10:-10], V_eff[10:-10], 'b-', lw=2)
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$V_{\\mathrm{eff}}(x)$')
axes[0].set_title('Effective potential from invariant measure')
axes[0].set_ylim(-5, 50)

x_schrod = x_fine[10:-10]
V_eff_clean = V_eff[10:-10]
V_eff_interp = lambda x: np.interp(x, x_schrod, V_eff_clean)
E_eff, psi_eff = solve_eigenstates(V_eff_interp, x_schrod, n_states=10)

axes[1].bar(range(10), E_eff, color='steelblue')
axes[1].set_xlabel('State index'); axes[1].set_ylabel('$E_n$')
axes[1].set_title('Eigenvalues of effective Hamiltonian')

psi_ground = np.abs(psi_eff[:, 0]) / np.max(np.abs(psi_eff[:, 0]))
psi_target = np.sqrt(rho_inv[10:-10]) / np.max(np.sqrt(rho_inv[10:-10]))

axes[2].plot(x_schrod, psi_target, 'r-', lw=2, label=r'$\sqrt{\rho_{\mathrm{inv}}}$ (target)')
axes[2].plot(x_schrod, psi_ground, 'b--', lw=2, label='$\\psi_0$ of $H_{\\mathrm{eff}}$')
axes[2].set_xlabel('$x$'); axes[2].set_ylabel('$\\psi$')
axes[2].set_title('Self-consistency check')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Self-consistency fails: \u03c8_0 of H_eff does NOT match \u221a\u03c1_inv")
print(f"Reason: \u221a\u03c1 diverges at boundaries; finite-difference H can't capture this")

### Experiment 3 Summary

**3a: Frobenius-Perron operator** — All four predictions confirmed. The approach works but IS the Koopman/transfer-operator framework that already exists [Ruelle 1986, Mezic 2005].

**3b: Effective potential from invariant measure** — Self-consistency fails. $\sqrt{\rho_{\text{inv}}}$ diverges at $x=0,1$ (arcsine distribution), but $\psi_0$ of $H_{\text{eff}}$ must vanish at boundaries. The boundary conditions are incompatible.

## 7. Ruelle-Pollicott Resonances

### Experiments 4-5: RP Resonance Convergence and Parameter Sweep

**Known result for logistic map at $a=4$:** RP resonances are $\lambda_n = 1/2^n$, so $|\lambda_1| = 1/2 = e^{-\ln 2}$.

In [ ]:
import dask
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=1, memory_limit='2GB')
client = Client(cluster)
print(f"Dask dashboard: {client.dashboard_link}")

# Experiment 4: Resolution convergence
N_values = [50, 100, 200, 400, 800, 1200, 1600]
n_keep = 12
exact_rp = np.array([1.0] + [1.0 / 2**n for n in range(1, n_keep)])

t0 = time.perf_counter()
futures = [dask.delayed(fp_spectrum_at_params)(logistic, 4.0, N, n_keep=n_keep)
           for N in N_values]
spectra_list = dask.compute(*futures)
t_total = time.perf_counter() - t0
print(f"Computed {len(N_values)} resolutions in {t_total:.1f}s")

spectra = np.array(spectra_list)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for k in range(min(6, n_keep)):
    axes[0].plot(N_values, np.abs(spectra[:, k]), 'o-', ms=5, label=f'$|\\lambda_{k}|$')
    axes[0].axhline(exact_rp[k], ls='--', alpha=0.4, color='gray')
axes[0].set_xlabel('Resolution $N$'); axes[0].set_ylabel('$|\\lambda_k|$')
axes[0].set_title('RP resonance convergence'); axes[0].legend(fontsize=8); axes[0].set_xscale('log')

ev_best = spectra[-1, :]
axes[1].scatter(np.real(ev_best), np.imag(ev_best), s=40, c='steelblue')
for r in [0.5, 0.25, 0.125]:
    axes[1].plot(r*np.cos(theta), r*np.sin(theta), 'k--', alpha=0.2)
axes[1].set_xlabel('Re(\u03bb)'); axes[1].set_ylabel('Im(\u03bb)')
axes[1].set_title(f'FP spectrum at N={N_values[-1]}')
axes[1].set_aspect('equal'); axes[1].set_xlim(-1.2, 1.2); axes[1].set_ylim(-1.2, 1.2)

lambda1_error = np.abs(np.abs(spectra[:, 1]) - 0.5)
axes[2].loglog(N_values, lambda1_error, 'ro-', ms=6, lw=2)
axes[2].set_xlabel('Resolution $N$'); axes[2].set_ylabel('$||\\lambda_1| - 1/2|$')
axes[2].set_title('Convergence of leading RP resonance')

plt.tight_layout()
plt.show()

print(f"\nBest estimate (N={N_values[-1]}): |\u03bb\u2081| = {np.abs(spectra[-1, 1]):.6f} (exact: 0.5)")

In [ ]:
# Experiment 5: RP resonances across the chaotic regime
a_sweep = np.linspace(3.5, 4.0, 120)
N_hi = 400
n_keep_sweep = 10

t0 = time.perf_counter()
futures_sweep = [dask.delayed(fp_spectrum_at_params)(logistic, a, N_hi, n_keep=n_keep_sweep)
                 for a in a_sweep]
spectra_sweep = np.array(dask.compute(*futures_sweep))
t_sweep = time.perf_counter() - t0
print(f"Computed {len(a_sweep)} parameter values in {t_sweep:.1f}s")

# Lyapunov exponent computation
lyap_est = np.zeros(len(a_sweep))
for i, a_val in enumerate(a_sweep):
    x = 0.5
    for _ in range(500): x = a_val * x * (1 - x)
    s = 0.0
    for _ in range(2000):
        x = a_val * x * (1 - x)
        deriv = abs(a_val * (1 - 2*x))
        if deriv > 1e-15: s += np.log(deriv)
    lyap_est[i] = s / 2000

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for k in range(min(6, n_keep_sweep)):
    axes[0,0].scatter(a_sweep, np.abs(spectra_sweep[:, k]), s=4, alpha=0.7)
axes[0,0].set_xlabel('$a$'); axes[0,0].set_ylabel('$|\\lambda_k|$')
axes[0,0].set_title('RP resonance magnitudes'); axes[0,0].set_ylim(0, 1.15)

axes[0,1].plot(a_sweep, 1.0 - np.abs(spectra_sweep[:, 1]), 'b-', lw=1)
axes[0,1].set_xlabel('$a$'); axes[0,1].set_ylabel('Spectral gap')
axes[0,1].set_title('Spectral gap (correlation decay rate)')

axes[1,0].scatter(a_sweep, np.angle(spectra_sweep[:, 1]), s=4, alpha=0.7, c='steelblue')
axes[1,0].set_xlabel('$a$'); axes[1,0].set_ylabel('arg($\\lambda_1$)')
axes[1,0].set_title('Phase of leading subeigenvalue')

chaotic_mask = lyap_est > 0.01
exp_neg_lyap = np.exp(-lyap_est)
lambda1_mag = np.abs(spectra_sweep[:, 1])
axes[1,1].scatter(exp_neg_lyap[chaotic_mask], lambda1_mag[chaotic_mask],
                  s=8, alpha=0.6, c=a_sweep[chaotic_mask], cmap='viridis')
axes[1,1].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='$|\\lambda_1| = e^{-\\lambda}$')
axes[1,1].set_xlabel('$e^{-\\lambda_{Lyap}}$'); axes[1,1].set_ylabel('$|\\lambda_1|$')
axes[1,1].set_title('RP resonance vs Lyapunov exponent')
axes[1,1].legend()

corr = np.corrcoef(exp_neg_lyap[chaotic_mask], lambda1_mag[chaotic_mask])[0, 1]
print(f"Correlation |\u03bb\u2081| vs e^(-\u03bb_Lyap): r = {corr:.4f}")

plt.tight_layout()
plt.show()

client.close()
cluster.close()
print("Dask cluster shut down.")

### Experiments 4-5 Summary

**Exp 4:** Slow, non-monotonic convergence of $|\lambda_1|$ toward $1/2$. The Monte Carlo binning introduces stochastic noise. Sub-eigenvalue spectral separation requires more sophisticated discretization.

**Exp 5:** $|\lambda_1|$ correlates with $e^{-\lambda_{\text{Lyap}}}$ at $r = 0.88$, confirming the Ruelle identity qualitatively. The spectral gap structure across the Feigenbaum cascade is rich — zero in periodic regime, growing through chaos onset, with dips at periodic windows.

## 8. Singular Potentials

### Experiment 9: Does Riccati Handle Coulomb Better?

**Prediction:** The Riccati ODE $y' = y^2 + 2/r + 2E$ might bypass the $1/r$ singularity that plagues finite-difference methods, since it can be integrated starting *away* from the origin.

**Exact eigenvalues:** $E_n = -1/(2n^2)$ for hydrogen s-wave.

In [ ]:
n_states = 6
E_exact_coul = -1.0 / (2.0 * np.arange(1, n_states + 1)**2)
N_c = 800; r_max = 60.0

V_coulomb = lambda r: coulomb(r)

# Matrix diag at r_min = 0.01
r_grid_c = np.linspace(0.01, r_max, N_c)
t0 = time.perf_counter()
E_mat_c = np.sort(np.linalg.eigvalsh(discretize_hamiltonian(V_coulomb, r_grid_c)))
t_mat_c = time.perf_counter() - t0
E_bound = E_mat_c[E_mat_c < 0][:n_states]

# Numerov
E_scan_c = np.linspace(-0.55, -0.001, 2000)
t0 = time.perf_counter()
E_num_c = numerov_detect(V_coulomb, E_scan_c, r_grid_c, order=15)
t_num_c = time.perf_counter() - t0
E_num_c = E_num_c[E_num_c < 0][:n_states]

# Riccati
t0 = time.perf_counter()
E_ric_c = detect_eigenvalues(V_coulomb, E_scan_c, x_span=(0.05, r_max), order=15)
t_ric_c = time.perf_counter() - t0
E_ric_c = E_ric_c[E_ric_c < 0][:n_states]

print("Coulomb Potential Benchmark:")
print(f"  Matrix: {len(E_bound)} states, {t_mat_c:.3f}s")
print(f"  Numerov: {len(E_num_c)} states, {t_num_c:.3f}s")
print(f"  Riccati: {len(E_ric_c)} states, {t_ric_c:.3f}s")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sensitivity to r_min
for n_idx in range(min(3, n_states)):
    errs = []
    rms = [0.01, 0.05, 0.1, 0.5]
    for rm in rms:
        rg = np.linspace(rm, r_max, N_c)
        em = np.sort(np.linalg.eigvalsh(discretize_hamiltonian(V_coulomb, rg)))
        eb = em[em < 0]
        errs.append(abs(eb[n_idx] - E_exact_coul[n_idx]) if n_idx < len(eb) else np.nan)
    axes[0].semilogy(rms, errs, 'o-', label=f'n={n_idx+1}')
axes[0].set_xlabel('$r_{min}$'); axes[0].set_ylabel('Error')
axes[0].set_title('Matrix: sensitivity to $r_{min}$'); axes[0].legend()

# Shooting function
for parity, color in [('even', 'C0'), ('odd', 'C1')]:
    psi_end = np.array([numerov_shoot(V_coulomb, E, r_grid_c, parity)[-1]
                        for E in E_scan_c[:500]])
    axes[1].plot(E_scan_c[:500], np.log10(np.abs(psi_end) + 1e-15), color=color, alpha=0.5)
for e in E_exact_coul: axes[1].axvline(e, color='red', ls='--', alpha=0.5)
axes[1].set_xlabel('E'); axes[1].set_ylabel('$\\log_{10}|\\psi(r_{max})|$')
axes[1].set_title('Numerov shooting function'); axes[1].set_ylim(-15, 5)

# Riccati trajectory near E_1
for dE, ls in [(0.0, '-'), (0.01, '--'), (-0.01, ':')]:
    E_test = E_exact_coul[0] + dE
    sol = riccati_solve(V_coulomb, E_test, (0.05, r_max), y0=0.5)
    axes[2].plot(sol.t, sol.y[0], ls, label=f'E={E_test:.3f}')
axes[2].set_xlabel('r'); axes[2].set_ylabel('y(r)')
axes[2].set_title('Riccati trajectory near $E_1$'); axes[2].legend()
axes[2].set_ylim(-10, 10)

plt.suptitle('Experiment 9: Coulomb Singular Potential', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nVerdict: Riccati ~{t_ric_c/t_mat_c:.0f}x slower than matrix, no accuracy gain.")
print(f"The y\u00b2 nonlinearity amplifies the singularity rather than bypassing it.")

## 9. Transfer Operator Discretization

### Experiment 10: Monte Carlo vs Ulam vs Chebyshev Spectral

**Key prediction:** Chebyshev spectral methods should give exponential convergence for smooth problems but may fail when eigenfunctions are singular — as they are for chaotic maps.

In [ ]:
a_test_10 = 4.0
n_keep_10 = 6
exact_rp_10 = np.array([1.0] + [1.0/2**n for n in range(1, n_keep_10)])

N_vals_10 = [50, 100, 200, 400, 800, 1600]

mc_spectra = {}
ulam_spectra = {}

for N_10 in N_vals_10:
    L_mc, _ = frobenius_perron_matrix(logistic, a_test_10, N=N_10)
    ev = np.linalg.eigvals(L_mc)
    mc_spectra[N_10] = ev[np.argsort(np.abs(ev))[::-1][:n_keep_10]]

    L_ul, _ = exact_ulam_matrix(logistic, a_test_10, N=N_10, n_sub=200)
    ev = np.linalg.eigvals(L_ul)
    ulam_spectra[N_10] = ev[np.argsort(np.abs(ev))[::-1][:n_keep_10]]

# Chebyshev (smaller N, expect failure)
cheb_spectra = {}
for N_10 in [16, 32, 64]:
    L_ch, _ = chebyshev_transfer_matrix(logistic, a_test_10, N=N_10)
    ev = np.linalg.eigvals(L_ch)
    cheb_spectra[N_10] = ev[np.argsort(np.abs(ev))[::-1][:n_keep_10]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

N_arr = np.array(N_vals_10)
lam1_mc = [abs(abs(mc_spectra[N_10][1]) - 0.5) for N_10 in N_vals_10]
lam1_ul = [abs(abs(ulam_spectra[N_10][1]) - 0.5) for N_10 in N_vals_10]
axes[0].loglog(N_arr, lam1_mc, 'o-', label='MC (50 sub)')
axes[0].loglog(N_arr, lam1_ul, 's-', label='Ulam (200 sub)')
axes[0].loglog(N_arr, 0.5/np.sqrt(N_arr), 'k--', alpha=0.3, label='$O(1/\\sqrt{N})$')
axes[0].set_xlabel('N'); axes[0].set_ylabel('$||\\lambda_1| - 0.5|$')
axes[0].set_title('Convergence of $|\\lambda_1|$'); axes[0].legend(fontsize=8)

axes[1].semilogy(range(n_keep_10), exact_rp_10, 'k^', ms=10, label='Exact $1/2^n$')
axes[1].semilogy(range(n_keep_10), np.abs(ulam_spectra[1600]), 'bs', ms=7, label='Ulam N=1600')
axes[1].semilogy(range(n_keep_10), np.abs(mc_spectra[1600]), 'ro', ms=7, label='MC N=1600')
axes[1].set_xlabel('Index'); axes[1].set_ylabel('$|\\lambda_n|$')
axes[1].set_title('RP spectrum at $a=4$'); axes[1].legend()

# Ulam sweep across chaos regime
a_sweep_10 = np.linspace(3.6, 4.0, 100)
lyap_sweep_10 = np.zeros(len(a_sweep_10))
ulam_sweep_10 = np.zeros((len(a_sweep_10), 5))
for i, a in enumerate(a_sweep_10):
    try:
        L, _ = exact_ulam_matrix(logistic, a, N=400, n_sub=200)
        ev = np.linalg.eigvals(L)
        ulam_sweep_10[i] = np.abs(ev[np.argsort(np.abs(ev))[::-1][:5]])
    except: ulam_sweep_10[i] = np.nan
    x_orb = 0.5
    for _ in range(500): x_orb = logistic(a, x_orb)
    s = 0.0
    for _ in range(10000):
        x_orb = logistic(a, x_orb)
        d = abs(a*(1-2*x_orb))
        if d > 1e-15: s += np.log(d)
    lyap_sweep_10[i] = s / 10000

for k in range(1, 5):
    axes[2].plot(a_sweep_10, ulam_sweep_10[:, k], '.', ms=3)
axes[2].plot(a_sweep_10, np.exp(-lyap_sweep_10), 'k-', lw=1.5, alpha=0.7, label='$e^{-\\lambda_{Lyap}}$')
axes[2].set_xlabel('$a$'); axes[2].set_ylabel('Magnitude')
axes[2].set_title('RP spectrum across chaos (Ulam N=400)'); axes[2].legend(fontsize=8)

plt.suptitle('Experiment 10: Improved Transfer Operator Discretization', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nFundamental obstacle: Chebyshev FAILS because the invariant density is singular.")
print(f"Polynomial bases cannot represent singular eigenfunctions.")
print(f"This is a genuine obstruction to bringing spectral methods to chaotic dynamics.")

## 10. Riccati Regularity Boundary

### Experiment 11: Quantization as Regularity

Scan the $(E, y_0)$ parameter plane and compute $\max|y(x)|$ over the Riccati trajectory. At eigenvalues, narrow channels of bounded solutions appear.

In [ ]:
# Harmonic oscillator regularity map
E_range_ho = np.linspace(0, 5.5, 400)
y0_range_ho = np.linspace(-3.0, 3.0, 300)

t0 = time.perf_counter()
reg_ho = riccati_regularity_map(harmonic, E_range_ho, y0_range_ho,
                                 x_span=(0.0, 8.0), max_y=50.0, n_x=500)
print(f"Regularity map computed in {time.perf_counter()-t0:.1f}s")

# Channel width analysis
print("\nChannel widths (harmonic oscillator):")
threshold = np.log10(10.0)
for n in range(6):
    E_exact_ho = n + 0.5
    j_E = np.argmin(np.abs(E_range_ho - E_exact_ho))
    col = reg_ho[:, j_E]
    width = np.sum(col < threshold) * (y0_range_ho[1] - y0_range_ho[0])
    i_best = np.argmin(col)
    y0_best = y0_range_ho[i_best]
    print(f"  n={n}: E={E_exact_ho:.1f}, best y\u2080={y0_best:+.3f}, width={width:.3f}")

# Double well
E_range_dw = np.linspace(-0.5, 8.0, 400)
y0_range_dw = np.linspace(-5.0, 5.0, 300)
reg_dw = riccati_regularity_map(double_well, E_range_dw, y0_range_dw,
                                 x_span=(0.0, 5.0), max_y=50.0, n_x=500)

x_ref_dw = np.linspace(-4, 4, 2000)
E_dw_exact, _ = solve_eigenstates(double_well, x_ref_dw, n_states=6)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].pcolormesh(E_range_ho, y0_range_ho, reg_ho,
                          cmap='inferno_r', vmin=-1, vmax=2, shading='auto')
plt.colorbar(im0, ax=axes[0], label='$\\log_{10}(\\max|y|)$')
for n in range(6): axes[0].axvline(n + 0.5, color='cyan', ls='--', alpha=0.7, lw=0.8)
axes[0].set_xlabel('Energy $E$'); axes[0].set_ylabel('$y_0$')
axes[0].set_title('Harmonic: $V = x^2/2$')

im1 = axes[1].pcolormesh(E_range_dw, y0_range_dw, reg_dw,
                          cmap='inferno_r', vmin=-1, vmax=2, shading='auto')
plt.colorbar(im1, ax=axes[1], label='$\\log_{10}(\\max|y|)$')
for e in E_dw_exact: axes[1].axvline(e, color='cyan', ls='--', alpha=0.7, lw=0.8)
axes[1].set_xlabel('Energy $E$'); axes[1].set_ylabel('$y_0$')
axes[1].set_title('Double Well: $V = (x^2-1)^2$')

plt.suptitle('Experiment 11: Riccati Regularity Boundary\nQuantization = global regularity of a nonlinear ODE',
             fontsize=13)
plt.tight_layout()
plt.show()

### Experiment 11 Summary

The regularity map beautifully visualizes **quantization as regularity**: eigenvalues are the discrete parameter values where a nonlinear ODE admits globally regular solutions. This is Turbiner's perspective made concrete.

Key findings:
- **Ground state channel** is wide ($\Delta y_0 \approx 2.0$) and centered at $y_0 = 0$, matching the exact $\psi_0 \propto e^{-x^2/2}$
- **Excited state channels** are exponentially thin — the $y^2$ nonlinearity amplifies IC perturbations through $n$ poles
- **Double-well tunneling split** is visible as two closely-spaced channels

**But computationally**, this is equivalent to shooting. The channel boundaries are exactly where $\psi(x_{\max}) = 0$.

## 11. Summary and Knowledge Gaps

### What We Tested

| Experiment | Question | Result |
|------------|----------|--------|
| 1 | Speed advantage for Riccati? | **No** — 1200x slower than matrix diag |
| 2 | $\alpha$-relaxation for eigenvalue detection? | **Wrong tradeoff** — sensitivity vs stability |
| 3a | Frobenius-Perron operator? | **Works** but is known territory (Ruelle, Koopman) |
| 3b | Effective potential from invariant measure? | **Fails** — boundary condition mismatch |
| 4-5 | Ruelle-Pollicott resonances? | **Confirmed** qualitatively; discretization limits quantitative test |
| 9 | Riccati on singular potentials? | **No advantage** — $y^2$ amplifies singularity |
| 10 | Spectral transfer operator discretization? | **Fundamental obstacle** — singular eigenfunctions |
| 11 | Regularity boundary structure? | **Beautiful** but computationally equivalent to shooting |

### Knowledge Gaps That Remain

1. **Better transfer operator discretization:** The Ulam method converges for the invariant measure but not for sub-leading Ruelle-Pollicott resonances [Blank-Keller-Liverani 2002]. Spectral methods fail due to singular eigenfunctions. Is there a basis that matches the singularity structure (e.g., Jacobi polynomials with endpoint weights)?

2. **Discrete Riccati analog:** Can we construct a "Riccati equation" directly for discrete maps, without going through V_eff? The continuous Riccati arises from the log-wavefunction substitution. Is there a discrete analog via the log-eigenfunction of the Perron-Frobenius operator?

3. **QES classification gaps:** Turbiner classified quasi-exactly-solvable potentials in 1D. Has this been extended to higher dimensions? Are there QES models relevant to condensed matter (e.g., periodic potentials, disordered systems)?

4. **Riccati in many-body physics:** The Riccati substitution turns the linear Schr\"odinger equation into a nonlinear ODE. For many-body problems, the Schr\"odinger equation is already intractable. Does the Riccati perspective offer insight into many-body methods (DMFT, DFT, tensor networks)?

5. **Connection to control theory:** The algebraic Riccati equation is central to LQR/Kalman filtering. Is there a precise connection between Turbiner's quantum Riccati and the control-theory Riccati beyond the shared name?

### Where This Leads

See [crossover_alpha_turbiner.ipynb](crossover_alpha_turbiner.ipynb) for the application of both the $\alpha$-transform and Turbiner's method to the Dyson equation, which is the first setting where the combined approach yields a genuinely useful (and possibly novel) result.